# Self-contained exporter for six interpretable Model 4C regimes

This notebook contains only the code needed to:

1. initialize the base Model 4C world;
2. simulate one named regime;
3. immediately render and save its dual-view MP4;
4. delete that regime from memory;
5. continue to the next regime.

It does **not** require the large FINAL notebook or any pre-saved `.pkl.gz` files.

In [1]:
# ============================================================
# Lightweight configuration
# ============================================================

from pathlib import Path
import gc
import re

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# Model size. These match the original notebook.
N = 100
M = 300
T = 100
K = 5

# Animation/export controls.
OUTPUT_DIR = Path("named_regime_mp4s")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FRAME_STEP = 4       # Increase to 8 if memory is still too high.
FPS = 8
DPI = 90             # Lower than the original to reduce memory.
INTERVAL_MS = 125
NODE_SIZE_MIN = 18
NODE_SIZE_MAX = 85
EDGE_HIGHLIGHT_FRAMES = 1

BASE_SEED = 42
np.random.seed(BASE_SEED)

# Use imageio-ffmpeg when available.
try:
    import imageio_ffmpeg
    plt.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()
except ImportError:
    pass

## Minimal model initialization functions

In [2]:
def initialize_discrete_opinions(N, p_positive=0.5):
    """
    Creates binary opinions in {-1, +1}.
    p_positive controls the initial proportion of +1 opinions.
    """
    return np.random.choice([-1, 1], size=N, p=[1 - p_positive, p_positive])


def initialize_continuous_opinions(N, distribution="bimodal"):
    """
    Creates continuous opinions in [-1, 1].

    distribution options:
    - uniform: opinions spread evenly across [-1, 1]
    - normal: opinions concentrated near 0
    - bimodal: two polarized initial clusters
    """
    if distribution == "uniform":
        opinions = np.random.uniform(-1, 1, N)

    elif distribution == "normal":
        opinions = np.random.normal(0, 0.35, N)

    elif distribution == "bimodal":
        group1 = np.random.normal(-0.6, 0.15, N // 2)
        group2 = np.random.normal(0.6, 0.15, N - N // 2)
        opinions = np.concatenate([group1, group2])
        np.random.shuffle(opinions)

    else:
        raise ValueError("distribution must be 'uniform', 'normal', or 'bimodal'")

    return np.clip(opinions, -1, 1)

In [3]:
def initialize_agent_traits(N):
    """
    Creates heterogeneous user-level traits.

    These are mainly used in the continuous user-content bipartite recommender model.
    Each trait is user-specific, so exposure and updating can depend on user i.
    """
    traits = {
        # Mostly low/moderate openness, with a few highly influenceable users
        "susceptibility": np.random.beta(2, 5, N),

        # Resistance to change / anchoring to original opinion
        "stubbornness": np.random.beta(2, 6, N),

        # Not every user updates every time step
        "activity": np.random.uniform(0.6, 1.0, N),

        # Tendency toward stronger versions of current opinion
        "extremity_bias": np.random.beta(2, 8, N),

        # User-specific confidence threshold: how far content can be before it stops feeling persuasive
        "epsilon": np.random.uniform(0.20, 0.50, N),

        # User-specific personalization strength: how strongly the recommender favors similar content
        "beta": np.random.uniform(2.0, 8.0, N)
    }
    return traits

In [4]:
def initialize_content(M, distribution="bimodal"):
    """
    Creates content positions y_j in [-1, 1].

    Content can be interpreted as posts, videos, articles, or recommended items.
    The position y_j represents the opinion, preference, or cultural location of content j.
    """
    if distribution == "uniform":
        content = np.random.uniform(-1, 1, M)

    elif distribution == "normal":
        content = np.random.normal(0, 0.35, M)

    elif distribution == "bimodal":
        group1 = np.random.normal(-0.65, 0.15, M // 2)
        group2 = np.random.normal(0.65, 0.15, M - M // 2)
        content = np.concatenate([group1, group2])
        np.random.shuffle(content)

    else:
        raise ValueError("distribution must be 'uniform', 'normal', or 'bimodal'")

    return np.clip(content, -1, 1)


def initialize_content_popularity(M):
    """
    Simulates content popularity / engagement potential.

    A heavy-tailed distribution is used because real online platforms often have
    a few very popular items and many low-visibility items.
    """
    popularity = np.random.pareto(a=2.0, size=M) + 1
    popularity = popularity / np.max(popularity)
    return popularity

In [5]:
def create_small_world_graph(N, k=6, rewiring_p=0.1, seed=42):
    """
    Creates a connected Watts-Strogatz small-world graph.

    N: number of users
    k: each node is initially connected to k nearest neighbors
    rewiring_p: probability of rewiring each edge

    Interpretation:
    clustered social circles with occasional long-range ties.
    """
    if k >= N:
        k = N - 1
    if k % 2 == 1:
        k += 1

    G = nx.watts_strogatz_graph(N, k=k, p=rewiring_p, seed=seed)

    attempt = 0
    while not nx.is_connected(G):
        attempt += 1
        G = nx.watts_strogatz_graph(N, k=k, p=rewiring_p, seed=seed + attempt)

    return G


def create_homophily_graph(s0, target_avg_degree=6, homophily_strength=4.0, seed=42):
    """
    Creates a connected graph where users with similar initial opinions
    are more likely to be connected.

    s0: initial opinions in [-1, 1]
    target_avg_degree: approximate social degree
    homophily_strength: larger values create stronger similarity-based sorting

    Interpretation:
    social connections are more likely among users who already share similar tastes.
    """
    rng = np.random.default_rng(seed)
    N = len(s0)

    G = nx.Graph()
    G.add_nodes_from(range(N))

    base_p = target_avg_degree / (N - 1)

    for i in range(N):
        for j in range(i + 1, N):
            distance = abs(s0[i] - s0[j])
            p_ij = base_p * np.exp(-homophily_strength * distance)

            if rng.random() < p_ij:
                G.add_edge(i, j)

    # Avoid isolated nodes by connecting each isolated user to the nearest opinion neighbor.
    for i in range(N):
        if G.degree[i] == 0:
            distances = np.abs(s0 - s0[i])
            distances[i] = np.inf
            nearest = int(np.argmin(distances))
            G.add_edge(i, nearest)

    # Connect disconnected components using nearest-opinion bridges.
    while not nx.is_connected(G):
        components = list(nx.connected_components(G))
        comp_a = list(components[0])
        comp_b = list(components[1])

        best_pair = None
        best_distance = np.inf

        for i in comp_a:
            for j in comp_b:
                d = abs(s0[i] - s0[j])
                if d < best_distance:
                    best_distance = d
                    best_pair = (i, j)

        G.add_edge(*best_pair)

    return G


def create_social_graph(
    graph_type,
    s0,
    N,
    target_avg_degree=6,
    small_world_rewiring=0.1,
    homophily_strength=4.0,
    seed=42
):
    """
    Wrapper for choosing the social graph type used in Model 4.
    """
    if graph_type == "small_world":
        return create_small_world_graph(
            N=N,
            k=target_avg_degree,
            rewiring_p=small_world_rewiring,
            seed=seed
        )

    elif graph_type == "homophily":
        return create_homophily_graph(
            s0=s0,
            target_avg_degree=target_avg_degree,
            homophily_strength=homophily_strength,
            seed=seed
        )

    else:
        raise ValueError("graph_type must be 'small_world' or 'homophily'")



def update_dynamic_homophily_graph(
    G,
    opinions,
    homophily_strength=4.0,
    rewiring_fraction=0.05,
    rng=None
):
    """
    Rewires part of an existing user-user graph using CURRENT opinions.

    Existing edges between dissimilar users are more likely to be removed,
    while new edges between similar users are more likely to be added.
    The number of edges is approximately preserved and the graph is kept connected.

    Parameters
    ----------
    G : networkx.Graph
        Current user-user social graph. It is modified in place.
    opinions : array-like
        Current user opinions in [-1, 1].
    homophily_strength : float
        Larger values create stronger preference for similar-opinion ties.
    rewiring_fraction : float
        Fraction of current edges considered for replacement at an update.
    rng : numpy.random.Generator or None
        Random number generator.

    Returns
    -------
    G : networkx.Graph
        The updated graph.
    changes : dict
        Counts of removed and added edges.
    """
    if rng is None:
        rng = np.random.default_rng()
    if not 0 <= rewiring_fraction <= 1:
        raise ValueError("rewiring_fraction must lie in [0, 1].")
    if homophily_strength < 0:
        raise ValueError("homophily_strength must be nonnegative.")

    opinions = np.asarray(opinions, dtype=float)
    n_edges = G.number_of_edges()
    if n_edges == 0 or rewiring_fraction == 0:
        return G, {"removed": 0, "added": 0}

    n_rewire = max(1, int(round(rewiring_fraction * n_edges)))

    # Bridges are protected so that rewiring does not disconnect the network.
    protected = {tuple(sorted(e)) for e in nx.bridges(G)}
    removable = [
        tuple(sorted(e)) for e in G.edges()
        if tuple(sorted(e)) not in protected
    ]

    removed = 0
    if removable:
        distances = np.array([
            abs(opinions[i] - opinions[j]) for i, j in removable
        ], dtype=float)

        # Dissimilar ties receive greater removal weight.
        weights = np.expm1(homophily_strength * distances)
        if not np.all(np.isfinite(weights)) or weights.sum() <= 0:
            weights = np.ones(len(removable), dtype=float)
        probabilities = weights / weights.sum()

        n_remove = min(n_rewire, len(removable))
        selected = rng.choice(
            len(removable), size=n_remove, replace=False, p=probabilities
        )
        for idx in np.atleast_1d(selected):
            i, j = removable[int(idx)]
            if G.has_edge(i, j):
                G.remove_edge(i, j)
                removed += 1

    # Add the same number of edges, favoring similar current opinions.
    added = 0
    if removed > 0:
        nonedges = list(nx.non_edges(G))
        if nonedges:
            similarities = np.array([
                np.exp(-homophily_strength * abs(opinions[i] - opinions[j]))
                for i, j in nonedges
            ], dtype=float)
            if not np.all(np.isfinite(similarities)) or similarities.sum() <= 0:
                similarities = np.ones(len(nonedges), dtype=float)
            probabilities = similarities / similarities.sum()

            n_add = min(removed, len(nonedges))
            selected = rng.choice(
                len(nonedges), size=n_add, replace=False, p=probabilities
            )
            for idx in np.atleast_1d(selected):
                i, j = nonedges[int(idx)]
                G.add_edge(i, j)
                added += 1

    return G, {"removed": removed, "added": added}


def mean_edge_opinion_distance(G, opinions):
    """Mean absolute opinion difference across current social edges."""
    if G.number_of_edges() == 0:
        return 0.0
    return float(np.mean([
        abs(opinions[i] - opinions[j]) for i, j in G.edges()
    ]))

## Unified recommender

In [6]:
# ============================================================
# Unified recommender model
# ============================================================

def compute_content_exposure_probabilities(
    user_opinion,
    content_positions,
    content_popularity,
    epsilon_i,
    beta_i,
    algorithm="unified",
    popularity_strength=0.5,
    exploration=0.05,
    popularity_offset=1e-6
):
    """
    Compute the probability that a user is exposed to each content item.

    Parameters
    ----------
    user_opinion : float
        Current opinion x_i(t) of the user.

    content_positions : array-like
        Opinion/taste position y_j of each content item.

    content_popularity : array-like
        Popularity p_j of each content item.

    epsilon_i : float
        User confidence threshold. This is accepted for compatibility
        with the full model, but it affects influence after exposure,
        not the exposure probability itself.

    beta_i : float
        Personalization strength. Larger values strongly favor content
        close to the user's current opinion.

    algorithm : {"random", "unified"}
        "random" gives every content item equal probability.
        "unified" combines similarity, popularity, and exploration.

    popularity_strength : float
        Popularity weight q. Larger values favor already-popular items.

    exploration : float
        Probability r of drawing from a uniform distribution.

    popularity_offset : float
        Small constant c that prevents log(0).

    Returns
    -------
    probabilities : np.ndarray
        Exposure probabilities over all content items.
    """
    content_positions = np.asarray(content_positions, dtype=float)
    content_popularity = np.asarray(content_popularity, dtype=float)

    M = len(content_positions)

    if M == 0:
        raise ValueError("content_positions cannot be empty.")

    if len(content_popularity) != M:
        raise ValueError(
            "content_positions and content_popularity must have equal length."
        )

    if not 0 <= exploration <= 1:
        raise ValueError("exploration must lie in [0, 1].")

    if beta_i < 0:
        raise ValueError("beta_i must be nonnegative.")

    if popularity_strength < 0:
        raise ValueError("popularity_strength must be nonnegative.")

    # --------------------------------------------------------
    # Random baseline
    # --------------------------------------------------------
    if algorithm == "random":
        return np.full(M, 1.0 / M)

    if algorithm != "unified":
        raise ValueError("algorithm must be either 'random' or 'unified'.")

    # --------------------------------------------------------
    # Unified ranking score
    # --------------------------------------------------------

    # Distance between the user's current opinion and every item.
    distances = np.abs(content_positions - user_opinion)

    # Similarity contribution:
    # close items receive higher scores.
    similarity_component = -beta_i * distances

    # Popularity contribution:
    # log scaling prevents the most popular item from completely
    # overwhelming the other items.
    safe_popularity = np.maximum(
        content_popularity,
        popularity_offset
    )

    popularity_component = (
        popularity_strength
        * np.log(safe_popularity + popularity_offset)
    )

    # Unified log score:
    #
    # score_ij = -beta_i |x_i - y_j| + q log(p_j + c)
    log_scores = similarity_component + popularity_component

    # Numerical stability before exponentiation.
    log_scores = log_scores - np.max(log_scores)

    ranking_scores = np.exp(log_scores)

    if (
        ranking_scores.sum() <= 0
        or not np.all(np.isfinite(ranking_scores))
    ):
        ranked_probabilities = np.full(M, 1.0 / M)
    else:
        ranked_probabilities = (
            ranking_scores / ranking_scores.sum()
        )

    # Exploration mixture:
    #
    # P = (1-r) P_ranked + r P_uniform
    uniform_probabilities = np.full(M, 1.0 / M)

    probabilities = (
        (1 - exploration) * ranked_probabilities
        + exploration * uniform_probabilities
    )

    # Final normalization protects against floating-point drift.
    probabilities = probabilities / probabilities.sum()

    return probabilities

## Hybrid social-recommender model with dynamic homophily and stubbornness

In [7]:
def compute_social_influence(i, s, G, epsilon_i):
    """
    Computes bounded-confidence user-user social influence on user i.

    User i is influenced by neighboring users only if their opinions are
    within epsilon_i of user i's current opinion.
    """
    neighbors = list(G.neighbors(i))

    if len(neighbors) == 0:
        return 0.0

    neighbor_opinions = s[neighbors]
    distances = np.abs(neighbor_opinions - s[i])
    confidence_mask = distances <= epsilon_i

    if not np.any(confidence_mask):
        return 0.0

    accepted_neighbors = neighbor_opinions[confidence_mask]
    return np.mean(accepted_neighbors - s[i])


def run_hybrid_social_recommender_model(
    s0,
    content_positions,
    content_popularity,
    social_graph,
    T,
    traits,
    K=5,
    algorithm="engagement",
    mu_content=0.25,
    mu_social=0.10,
    popularity_strength=0.5,
    exploration=0.05,
    amplification=0.12,
    noise_std=0.005,
    anchor_strength=0.05,
    dynamic_stubbornness=False,
    stability_threshold=0.05,
    consolidation_strength=0.45,
    consolidation_timescale=25.0,
    max_stubbornness=0.98,
    worldview_memory_rate=0.10,
    dynamic_homophily=False,
    homophily_rewire_interval=10,
    homophily_rewiring_fraction=0.05,
    dynamic_homophily_strength=4.0,
    graph_seed=42,
    return_interactions=False,
    return_diagnostics=False
):
    """
    Model 4: hybrid social-recommender opinion dynamics model.

    Users have continuous opinions s_i(t) in [-1, 1]. They update from two layers:

    1. User-content recommendation layer:
       users are exposed to algorithmically selected content items.

    2. User-user social layer:
       users are influenced by neighbors in a social graph.

    This one function reproduces the 4A/4B/4C progression:
    - 4A content-only: mu_content > 0, mu_social = 0
    - 4B social-only:  mu_content = 0, mu_social > 0
    - 4C hybrid:       mu_content > 0, mu_social > 0

    Optional dynamic stubbornness:
    - Each user has a moving worldview center.
    - If the user's opinion remains within stability_threshold of that center,
      residence time increases.
    - Stubbornness rises with residence time according to a saturating curve.
    - If the user moves outside the stable region, a new worldview center begins.

    Optional dynamic homophily:
    - The social graph can be rewired every homophily_rewire_interval steps.
    - Dissimilar current-opinion ties are preferentially removed.
    - Similar current-opinion pairs are preferentially connected.
    - The number of social edges is approximately preserved.

    Both dynamic_stubbornness and dynamic_homophily are optional so the
    content-only, social-only, and hybrid cases remain directly comparable.
    """
    if consolidation_timescale <= 0:
        raise ValueError("consolidation_timescale must be positive.")
    if not 0 <= worldview_memory_rate <= 1:
        raise ValueError("worldview_memory_rate must lie in [0, 1].")
    if homophily_rewire_interval <= 0:
        raise ValueError("homophily_rewire_interval must be positive.")
    if not 0 <= homophily_rewiring_fraction <= 1:
        raise ValueError("homophily_rewiring_fraction must lie in [0, 1].")

    N = len(s0)
    M = len(content_positions)

    history = np.zeros((T + 1, N))
    history[0] = s0.copy()

    s = s0.copy()
    initial = s0.copy()
    interaction_log = []

    # Work with a private graph copy so repeated experiments do not mutate
    # the same graph object across conditions.
    social_graph = social_graph.copy()
    graph_rng = np.random.default_rng(graph_seed)

    # Save graph snapshots for animation. A copy is required so that each
    # entry preserves the network structure at that specific time step.
    graph_history = {0: social_graph.copy()}

    # Dynamic-stubbornness state variables.
    base_stubbornness = np.asarray(traits["stubbornness"], dtype=float).copy()
    dynamic_stubbornness_values = base_stubbornness.copy()
    worldview_center = s0.copy()
    residence_time = np.zeros(N, dtype=int)

    stubbornness_history = np.zeros((T + 1, N))
    worldview_history = np.zeros((T + 1, N))
    residence_history = np.zeros((T + 1, N))

    stubbornness_history[0] = dynamic_stubbornness_values
    worldview_history[0] = worldview_center
    residence_history[0] = residence_time

    edge_distance_history = np.zeros(T + 1)
    edge_count_history = np.zeros(T + 1, dtype=int)
    clustering_history = np.zeros(T + 1)
    rewiring_history = np.zeros(T + 1, dtype=int)

    edge_distance_history[0] = mean_edge_opinion_distance(social_graph, s)
    edge_count_history[0] = social_graph.number_of_edges()
    clustering_history[0] = nx.average_clustering(social_graph)

    for t in range(T):
        new_s = s.copy()

        for i in range(N):
            # Activity: not every user actively processes new influence each step.
            if np.random.rand() > traits["activity"][i]:
                continue

            epsilon_i = traits["epsilon"][i]
            beta_i = traits["beta"][i]
            susceptibility_i = traits["susceptibility"][i]
            extremity_bias_i = traits["extremity_bias"][i]

            # -----------------------------
            # 1. Content/recommender influence
            # -----------------------------
            content_influence = 0.0

            if mu_content > 0:
                probabilities = compute_content_exposure_probabilities(
                    user_opinion=s[i],
                    content_positions=content_positions,
                    content_popularity=content_popularity,
                    epsilon_i=epsilon_i,
                    beta_i=beta_i,
                    algorithm=algorithm,
                    popularity_strength=popularity_strength,
                    exploration=exploration
                )

                recommended_items = np.random.choice(
                    np.arange(M),
                    size=min(K, M),
                    replace=False,
                    p=probabilities
                )

                shown_positions = content_positions[recommended_items]
                shown_distances = np.abs(shown_positions - s[i])

                shown_weights = probabilities[recommended_items]
                if shown_weights.sum() <= 0 or not np.all(np.isfinite(shown_weights)):
                    shown_weights = np.ones_like(shown_weights)
                shown_weights = shown_weights / shown_weights.sum()

                confidence_mask = shown_distances <= epsilon_i

                if np.any(confidence_mask):
                    content_influence = np.sum(
                        shown_weights[confidence_mask]
                        * (shown_positions[confidence_mask] - s[i])
                    )

                if return_interactions:
                    interaction_log.append({
                        "time": t,
                        "user": i,
                        "user_opinion": s[i],
                        "items": recommended_items,
                        "item_positions": shown_positions
                    })

            # -----------------------------
            # 2. Social/user-user influence
            # -----------------------------
            social_influence = 0.0

            if mu_social > 0:
                social_influence = compute_social_influence(
                    i=i,
                    s=s,
                    G=social_graph,
                    epsilon_i=epsilon_i
                )

            # -----------------------------
            # 3. Other terms
            # -----------------------------
            amplification_term = (
                amplification
                * extremity_bias_i
                * s[i]
                * (1 - abs(s[i]))
            )

            if dynamic_stubbornness:
                anchor_target = worldview_center[i]
                stubbornness_i = dynamic_stubbornness_values[i]
            else:
                anchor_target = initial[i]
                stubbornness_i = base_stubbornness[i]

            anchoring_term = (
                anchor_strength
                * stubbornness_i
                * (anchor_target - s[i])
            )

            noise = np.random.normal(0, noise_std)

            # -----------------------------
            # 4. Final opinion update
            # -----------------------------
            new_s[i] = (
                s[i]
                + mu_content * susceptibility_i * content_influence
                + mu_social * susceptibility_i * social_influence
                + amplification_term
                + anchoring_term
                + noise
            )


        new_s = np.clip(new_s, -1, 1)

        # -----------------------------
        # 5. Update worldview consolidation
        # -----------------------------
        if dynamic_stubbornness:
            stable = np.abs(new_s - worldview_center) <= stability_threshold

            residence_time[stable] += 1

            # A stable worldview center follows the user's nearby opinion slowly.
            worldview_center[stable] = (
                (1 - worldview_memory_rate) * worldview_center[stable]
                + worldview_memory_rate * new_s[stable]
            )

            # A large move begins residence in a new worldview region.
            changed_region = ~stable
            residence_time[changed_region] = 0
            worldview_center[changed_region] = new_s[changed_region]

            dynamic_stubbornness_values = np.clip(
                base_stubbornness
                + consolidation_strength
                * (1 - np.exp(-residence_time / consolidation_timescale)),
                0,
                max_stubbornness
            )
        else:
            dynamic_stubbornness_values = base_stubbornness.copy()

        # -----------------------------
        # 6. Dynamic homophily / network co-evolution
        # -----------------------------
        rewired_edges = 0
        if (
            dynamic_homophily
            and mu_social > 0
            and (t + 1) % homophily_rewire_interval == 0
        ):
            social_graph, graph_changes = update_dynamic_homophily_graph(
                G=social_graph,
                opinions=new_s,
                homophily_strength=dynamic_homophily_strength,
                rewiring_fraction=homophily_rewiring_fraction,
                rng=graph_rng
            )
            rewired_edges = graph_changes["added"]

            # Store the rewired graph at the opinion time t + 1.
            graph_history[t + 1] = social_graph.copy()

        s = new_s
        history[t + 1] = s.copy()
        stubbornness_history[t + 1] = dynamic_stubbornness_values
        worldview_history[t + 1] = worldview_center
        residence_history[t + 1] = residence_time
        edge_distance_history[t + 1] = mean_edge_opinion_distance(social_graph, s)
        edge_count_history[t + 1] = social_graph.number_of_edges()
        clustering_history[t + 1] = nx.average_clustering(social_graph)
        rewiring_history[t + 1] = rewired_edges

    diagnostics = {
        "stubbornness_history": stubbornness_history,
        "worldview_history": worldview_history,
        "residence_history": residence_history,
        "edge_distance_history": edge_distance_history,
        "edge_count_history": edge_count_history,
        "clustering_history": clustering_history,
        "rewiring_history": rewiring_history,
        "graph_history": graph_history,
        "final_social_graph": social_graph.copy()
    }

    if return_interactions and return_diagnostics:
        return history, interaction_log, diagnostics
    if return_interactions:
        return history, interaction_log
    if return_diagnostics:
        return history, diagnostics

    return history

## Heterogeneous parameter sampling

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product

#helpers
def copy_traits(traits):
    """
    Copy a dictionary whose values are NumPy arrays.
    """
    return {
        key: np.asarray(value).copy()
        for key, value in traits.items()
    }


def sample_scaled_beta(
    mean,
    maximum,
    size,
    concentration=12,
    rng=None
):
    """
    Sample a bounded heterogeneous population from a scaled Beta distribution.

    Parameters
    ----------
    mean : float
        Desired population mean in the interval [0, maximum].

    maximum : float
        Upper bound of the parameter.

    size : int
        Number of users.

    concentration : float
        Controls population heterogeneity.

        Larger concentration:
            users are more similar to one another.

        Smaller concentration:
            users are more heterogeneous.

    rng : np.random.Generator
        Optional random-number generator.
    """
    if rng is None:
        rng = np.random.default_rng()

    if maximum <= 0:
        raise ValueError("maximum must be positive.")

    normalized_mean = mean / maximum

    # Avoid invalid Beta parameters at exactly zero or one.
    normalized_mean = np.clip(
        normalized_mean,
        1e-4,
        1 - 1e-4
    )

    alpha = normalized_mean * concentration
    beta = (1 - normalized_mean) * concentration

    return maximum * rng.beta(alpha, beta, size=size)


def sample_personalization_strength(
    mean_beta,
    size,
    shape=8,
    maximum=12,
    rng=None
):
    """
    Sample heterogeneous personalization strengths from a Gamma distribution.

    The Gamma distribution is positive and right-skewed, allowing most users
    to have moderate personalization while a smaller number experience
    especially strong personalization.
    """
    if rng is None:
        rng = np.random.default_rng()

    if mean_beta < 0:
        raise ValueError("mean_beta must be nonnegative.")

    if mean_beta == 0:
        return np.zeros(size)

    scale = mean_beta / shape

    values = rng.gamma(
        shape=shape,
        scale=scale,
        size=size
    )

    return np.clip(values, 0, maximum)

## Six interpretable regimes

In [9]:
# ============================================================
# Interpretable Model 4C regimes
# ============================================================

named_regimes = {
    "Diverse weak-personalization environment": {
        "mean_beta": 1.0,
        "popularity_strength": 0.20,
        "exploration": 0.30,
        "mu_content": 0.20,
        "mu_social": 0.10,
        "mean_epsilon": 0.75,
        "mean_susceptibility": 0.40,
        "homophily_strength": 0.5,
        "rewiring_fraction": 0.01,
        "rewire_interval": 25,
        "dynamic_stubbornness": False,
        "consolidation_strength": 0.0
    },

    "Moderately personalized environment": {
        "mean_beta": 4.0,
        "popularity_strength": 0.50,
        "exploration": 0.10,
        "mu_content": 0.25,
        "mu_social": 0.15,
        "mean_epsilon": 0.50,
        "mean_susceptibility": 0.35,
        "homophily_strength": 2.0,
        "rewiring_fraction": 0.03,
        "rewire_interval": 15,
        "dynamic_stubbornness": True,
        "consolidation_strength": 0.30
    },

    "Algorithmic filter bubble": {
        "mean_beta": 8.0,
        "popularity_strength": 0.50,
        "exploration": 0.02,
        "mu_content": 0.25,
        "mu_social": 0.20,
        "mean_epsilon": 0.30,
        "mean_susceptibility": 0.30,
        "homophily_strength": 5.0,
        "rewiring_fraction": 0.08,
        "rewire_interval": 10,
        "dynamic_stubbornness": True,
        "consolidation_strength": 0.45
    },

    "Popularity-dominated mainstream": {
        "mean_beta": 2.0,
        "popularity_strength": 1.50,
        "exploration": 0.10,
        "mu_content": 0.35,
        "mu_social": 0.10,
        "mean_epsilon": 0.65,
        "mean_susceptibility": 0.45,
        "homophily_strength": 1.0,
        "rewiring_fraction": 0.02,
        "rewire_interval": 20,
        "dynamic_stubbornness": False,
        "consolidation_strength": 0.0
    },

    "Socially dominated echo chamber": {
        "mean_beta": 3.0,
        "popularity_strength": 0.30,
        "exploration": 0.10,
        "mu_content": 0.10,
        "mu_social": 0.35,
        "mean_epsilon": 0.30,
        "mean_susceptibility": 0.35,
        "homophily_strength": 6.0,
        "rewiring_fraction": 0.10,
        "rewire_interval": 10,
        "dynamic_stubbornness": True,
        "consolidation_strength": 0.50
    },

    "Cross-cutting high-openness environment": {
        "mean_beta": 1.0,
        "popularity_strength": 0.20,
        "exploration": 0.40,
        "mu_content": 0.45,
        "mu_social": 0.05,
        "mean_epsilon": 1.00,
        "mean_susceptibility": 0.65,
        "homophily_strength": 0.5,
        "rewiring_fraction": 0.01,
        "rewire_interval": 25,
        "dynamic_stubbornness": False,
        "consolidation_strength": 0.0
    }
}

## Build the shared initial world

In [10]:
# ============================================================
# Shared starting conditions across all six regimes
# ============================================================

np.random.seed(BASE_SEED)

s0_model4 = initialize_continuous_opinions(
    N,
    distribution="bimodal"
)

traits_model4 = initialize_agent_traits(N)

content_positions_model4 = initialize_content(
    M,
    distribution="bimodal"
)

content_popularity_model4 = initialize_content_popularity(M)

G_social = create_social_graph(
    graph_type="small_world",
    s0=s0_model4,
    N=N,
    target_avg_degree=6,
    small_world_rewiring=0.1,
    homophily_strength=4.0,
    seed=BASE_SEED
)

print("Users:", N)
print("Content items:", M)
print("Time steps:", T)
print("Initial social edges:", G_social.number_of_edges())

Users: 100
Content items: 300
Time steps: 100
Initial social edges: 300


## Run one named regime

In [11]:
def run_named_regime(
    regime_name,
    parameters,
    run_id=0,
    base_seed=20000
):
    """Run one representative realization of a named regime."""
    seed = (
        base_seed
        + 1000 * run_id
        + sum(ord(character) for character in regime_name)
    )

    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    traits_run = copy_traits(traits_model4)

    traits_run["epsilon"] = sample_scaled_beta(
        mean=parameters["mean_epsilon"],
        maximum=2.0,
        size=N,
        concentration=12,
        rng=rng
    )

    traits_run["susceptibility"] = sample_scaled_beta(
        mean=parameters["mean_susceptibility"],
        maximum=1.0,
        size=N,
        concentration=10,
        rng=rng
    )

    traits_run["beta"] = sample_personalization_strength(
        mean_beta=parameters["mean_beta"],
        size=N,
        shape=8,
        maximum=12,
        rng=rng
    )

    history, diagnostics = run_hybrid_social_recommender_model(
        s0=s0_model4.copy(),
        content_positions=content_positions_model4.copy(),
        content_popularity=content_popularity_model4.copy(),
        social_graph=G_social.copy(),
        T=T,
        traits=traits_run,
        K=K,

        algorithm="unified",
        popularity_strength=parameters["popularity_strength"],
        exploration=parameters["exploration"],

        mu_content=parameters["mu_content"],
        mu_social=parameters["mu_social"],

        dynamic_homophily=True,
        homophily_rewire_interval=parameters["rewire_interval"],
        homophily_rewiring_fraction=parameters["rewiring_fraction"],
        dynamic_homophily_strength=parameters["homophily_strength"],

        dynamic_stubbornness=parameters["dynamic_stubbornness"],
        consolidation_strength=parameters["consolidation_strength"],

        graph_seed=seed,
        return_diagnostics=True
    )

    return history, diagnostics

## Minimal animation helpers

In [12]:
def safe_filename(name):
    slug = re.sub(r"[^A-Za-z0-9]+", "_", name.strip()).strip("_").lower()
    return slug or "regime"


def canonical_edge(edge):
    u, v = edge
    return (u, v) if u < v else (v, u)


def active_graph_snapshot_time(graph_history, time_step):
    valid_times = [
        saved_time
        for saved_time in graph_history
        if saved_time <= time_step
    ]
    return max(valid_times)


def previous_graph_snapshot_time(graph_history, time_step):
    current_time = active_graph_snapshot_time(
        graph_history,
        time_step
    )
    earlier_times = [
        saved_time
        for saved_time in graph_history
        if saved_time < current_time
    ]
    return max(earlier_times) if earlier_times else None


def edge_changes_at_time(graph_history, time_step):
    current_time = active_graph_snapshot_time(
        graph_history,
        time_step
    )
    current_graph = graph_history[current_time]

    previous_time = previous_graph_snapshot_time(
        graph_history,
        time_step
    )

    current_edges = {
        canonical_edge(edge)
        for edge in current_graph.edges()
    }

    if previous_time is None:
        return current_graph, None, current_edges, set(), set()

    previous_graph = graph_history[previous_time]
    previous_edges = {
        canonical_edge(edge)
        for edge in previous_graph.edges()
    }

    return (
        current_graph,
        previous_graph,
        current_edges & previous_edges,
        current_edges - previous_edges,
        previous_edges - current_edges
    )


def mean_pairwise_distance(opinions):
    opinions = np.asarray(opinions)
    distances = np.abs(
        opinions[:, None] - opinions[None, :]
    )
    return float(np.mean(
        distances[np.triu_indices(len(opinions), k=1)]
    ))


def live_metrics(history, graph, time_step):
    opinions = history[time_step]
    initial = history[0]

    polarization = (
        mean_pairwise_distance(opinions)
        * np.mean(np.abs(opinions))
    )

    valid_initial = np.abs(initial) > 0.02
    opposite_side = (
        valid_initial
        & (initial * opinions < 0)
    )

    edge_distance = (
        np.mean([
            abs(opinions[i] - opinions[j])
            for i, j in graph.edges()
        ])
        if graph.number_of_edges() > 0
        else np.nan
    )

    return polarization, int(opposite_side.sum()), edge_distance


def make_frames(history_length):
    frames = list(range(0, history_length, FRAME_STEP))
    if frames[-1] != history_length - 1:
        frames.append(history_length - 1)
    return frames

In [13]:
def make_dual_view_animation(
    regime_name,
    history,
    diagnostics,
    layout_seed=42,
    jitter_seed=42
):
    graph_history = diagnostics["graph_history"]
    stubbornness_history = diagnostics["stubbornness_history"]

    # Compute layouts only for the current regime.
    union_graph = nx.Graph()
    union_graph.add_nodes_from(range(history.shape[1]))
    for snapshot in graph_history.values():
        union_graph.add_edges_from(snapshot.edges())

    spring_positions = nx.spring_layout(
        union_graph,
        seed=layout_seed,
        iterations=100
    )

    rng = np.random.default_rng(jitter_seed)
    y_jitter = rng.uniform(
        -0.20,
        0.20,
        size=history.shape[1]
    )

    frames = make_frames(history.shape[0])

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5.4)
    )

    norm = Normalize(vmin=-1, vmax=1)
    cmap = plt.get_cmap("coolwarm")

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])

    colorbar = fig.colorbar(
        sm,
        ax=axes,
        shrink=0.78,
        pad=0.02
    )
    colorbar.set_label("Opinion")

    def update(time_step):
        for ax in axes:
            ax.clear()

        opinions = history[time_step]
        stubbornness = stubbornness_history[time_step]

        sizes = (
            NODE_SIZE_MIN
            + stubbornness
            * (NODE_SIZE_MAX - NODE_SIZE_MIN)
        )

        (
            current_graph,
            previous_graph,
            persistent_edges,
            added_edges,
            removed_edges
        ) = edge_changes_at_time(
            graph_history,
            time_step
        )

        snapshot_time = active_graph_snapshot_time(
            graph_history,
            time_step
        )

        show_changes = (
            time_step - snapshot_time
            <= FRAME_STEP * EDGE_HIGHLIGHT_FRAMES
        )

        if not show_changes:
            persistent_edges = {
                canonical_edge(edge)
                for edge in current_graph.edges()
            }
            added_edges = set()
            removed_edges = set()
            previous_graph = None

        current_nodes = list(current_graph.nodes())
        node_colors = [
            opinions[node]
            for node in current_nodes
        ]
        node_sizes = [
            sizes[node]
            for node in current_nodes
        ]

        opinion_positions = {
            node: (opinions[node], y_jitter[node])
            for node in current_nodes
        }

        # Opinion-space view.
        nx.draw_networkx_edges(
            current_graph,
            opinion_positions,
            edgelist=list(persistent_edges),
            ax=axes[0],
            alpha=0.10,
            width=0.45,
            edge_color="gray"
        )

        if added_edges:
            nx.draw_networkx_edges(
                current_graph,
                opinion_positions,
                edgelist=list(added_edges),
                ax=axes[0],
                alpha=0.9,
                width=1.6,
                edge_color="green"
            )

        if previous_graph is not None and removed_edges:
            nx.draw_networkx_edges(
                previous_graph,
                opinion_positions,
                edgelist=list(removed_edges),
                ax=axes[0],
                alpha=0.65,
                width=1.1,
                edge_color="red",
                style="dashed"
            )

        nx.draw_networkx_nodes(
            current_graph,
            opinion_positions,
            ax=axes[0],
            nodelist=current_nodes,
            node_color=node_colors,
            cmap=cmap,
            vmin=-1,
            vmax=1,
            node_size=node_sizes,
            edgecolors="black",
            linewidths=0.12
        )

        axes[0].axvline(
            0,
            linestyle="--",
            linewidth=0.8,
            alpha=0.45
        )
        axes[0].set_xlim(-1.05, 1.05)
        axes[0].set_ylim(-0.32, 0.32)
        axes[0].set_yticks([])
        axes[0].set_xlabel("Opinion")
        axes[0].set_title("Opinion-space layout")

        # Fixed-network view.
        nx.draw_networkx_edges(
            current_graph,
            spring_positions,
            edgelist=list(persistent_edges),
            ax=axes[1],
            alpha=0.10,
            width=0.45,
            edge_color="gray"
        )

        if added_edges:
            nx.draw_networkx_edges(
                current_graph,
                spring_positions,
                edgelist=list(added_edges),
                ax=axes[1],
                alpha=0.9,
                width=1.6,
                edge_color="green"
            )

        if previous_graph is not None and removed_edges:
            nx.draw_networkx_edges(
                previous_graph,
                spring_positions,
                edgelist=list(removed_edges),
                ax=axes[1],
                alpha=0.65,
                width=1.1,
                edge_color="red",
                style="dashed"
            )

        nx.draw_networkx_nodes(
            current_graph,
            spring_positions,
            ax=axes[1],
            nodelist=current_nodes,
            node_color=node_colors,
            cmap=cmap,
            vmin=-1,
            vmax=1,
            node_size=node_sizes,
            edgecolors="black",
            linewidths=0.12
        )

        axes[1].set_title("Fixed network layout")
        axes[1].set_axis_off()

        polarization, opposite_side, edge_distance = live_metrics(
            history,
            current_graph,
            time_step
        )

        fig.suptitle(
            f"{regime_name} | Time = {time_step}\n"
            f"Polarization = {polarization:.3f} | "
            f"Opposite initial side = {opposite_side} | "
            f"Mean edge distance = {edge_distance:.3f}"
        )

        axes[0].text(
            0.02,
            0.03,
            "Color = opinion\n"
            "Size = stubbornness\n"
            "Green = added edge\n"
            "Red dashed = removed edge",
            transform=axes[0].transAxes,
            verticalalignment="bottom",
            fontsize=8,
            bbox={
                "boxstyle": "round",
                "facecolor": "white",
                "alpha": 0.80
            }
        )

    animation = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=INTERVAL_MS,
        repeat=False,
        cache_frame_data=False
    )

    return animation, fig

## Simulate and export all six regimes sequentially

The notebook never retains more than one regime and one animation in memory.
If an MP4 already exists, it is skipped so you can restart after a crash.

In [14]:
writer = FFMpegWriter(
    fps=FPS,
    metadata={
        "title": "Model 4C named-regime animation",
        "artist": "Virtual Earth project"
    },
    bitrate=1600
)

for index, (regime_name, parameters) in enumerate(
    named_regimes.items(),
    start=1
):
    output_path = (
        OUTPUT_DIR
        / f"{safe_filename(regime_name)}_dual_view.mp4"
    )

    if output_path.exists():
        print(
            f"[{index}/6] Skipping existing file: "
            f"{output_path.name}"
        )
        continue

    print(f"[{index}/6] Simulating: {regime_name}")

    history, diagnostics = run_named_regime(
        regime_name,
        parameters,
        run_id=0,
        base_seed=20000
    )

    print(f"    Rendering and saving: {output_path}")

    animation, figure = make_dual_view_animation(
        regime_name,
        history,
        diagnostics
    )

    animation.save(
        output_path,
        writer=writer,
        dpi=DPI
    )

    plt.close(figure)

    del animation
    del figure
    del history
    del diagnostics

    gc.collect()

    print("    Done.")

print("\nFinished. MP4 files are in:", OUTPUT_DIR.resolve())

[1/6] Simulating: Diverse weak-personalization environment
    Rendering and saving: named_regime_mp4s/diverse_weak_personalization_environment_dual_view.mp4
    Done.
[2/6] Simulating: Moderately personalized environment
    Rendering and saving: named_regime_mp4s/moderately_personalized_environment_dual_view.mp4
    Done.
[3/6] Simulating: Algorithmic filter bubble
    Rendering and saving: named_regime_mp4s/algorithmic_filter_bubble_dual_view.mp4
    Done.
[4/6] Simulating: Popularity-dominated mainstream
    Rendering and saving: named_regime_mp4s/popularity_dominated_mainstream_dual_view.mp4
    Done.
[5/6] Simulating: Socially dominated echo chamber
    Rendering and saving: named_regime_mp4s/socially_dominated_echo_chamber_dual_view.mp4
    Done.
[6/6] Simulating: Cross-cutting high-openness environment
    Rendering and saving: named_regime_mp4s/cross_cutting_high_openness_environment_dual_view.mp4
    Done.

Finished. MP4 files are in: /Users/yume/PycharmProjects/VirtualEarth/

## If PyCharm still becomes unresponsive

Use these more aggressive settings near the top:

```python
N = 70
M = 200
T = 80
FRAME_STEP = 8
DPI = 75
NODE_SIZE_MIN = 12
NODE_SIZE_MAX = 55
```

Changing `N`, `M`, or `T` changes the simulated system, so use those only if reducing
animation resolution and frame count is not enough. `FRAME_STEP` and `DPI` do not
change the simulation itself.